In [1]:
import os
import numpy as np
import rasterio
import tools
from dotenv import load_dotenv

load_dotenv()
krycklan_gis_data_path = os.getenv('KRYCKLAN_GIS_DATA_PATH')
print(f'Krycklan GIS data path: {krycklan_gis_data_path}')

# Filenames to process
base_masks = ["channels", "5haStreams"]

# Resolutions and catchments
resos = [20, 40, 80, 160]
catchment_no = np.arange(1, 23)
exclude = [11, 17, 18, 19]
catchs = np.setdiff1d(catchment_no, exclude)

for reso in resos:
    for catch in catchs:
        outdir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')

        for prefix in base_masks:
            # Construct file paths
            mask_file = os.path.join(outdir, f"{prefix}.asc")
            length_file = os.path.join(outdir, f"{prefix}_length.asc")
            depth_file = os.path.join(outdir, f"{prefix}_depth.asc")
            dist_file = os.path.join(outdir, f"{prefix}_distance.asc")
            width_file = os.path.join(outdir, f"{prefix}_width.asc")

            if os.path.exists(mask_file):
                out_file = os.path.join(outdir, f"{prefix}.asc")
                tools.update_mask(mask_file, length_file, depth_file, dist_file, width_file, out_file)
                print(f"✅ Cleaned mask written: {out_file}")

Krycklan GIS data path: /Users/jpnousu/Data/Krycklan_GIS_data
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/channels.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/5haStreams.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/channels.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/5haStreams.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C3/channels.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C3/5haStreams.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C4/channels.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C4/5haStreams.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C5/channels.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C5/5haStreams.asc
✅ Cleaned mask written: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C6/channels.asc
✅ Cleaned mask written: /U

In [2]:
info_suffixes = ["length", "depth", "distance", "width"]

for reso in resos:
    for catch in catchs:
        outdir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')

        for prefix in base_masks:
            mask_file = os.path.join(outdir, f"{prefix}.asc")

            if not os.path.exists(mask_file):
                continue

            with rasterio.open(mask_file) as mask_src:
                mask = mask_src.read(1)
                valid = mask != mask_src.nodata

            for suffix in info_suffixes:
                info_file = os.path.join(outdir, f"{prefix}_{suffix}.asc")

                if not os.path.exists(info_file):
                    continue

                with rasterio.open(info_file) as src:
                    data = src.read(1)
                    profile = src.profile.copy()
                    nodata_val = src.nodata if src.nodata is not None else -9999
                    profile.update(nodata=nodata_val)

                cleaned = np.where(valid, data, nodata_val)

                with rasterio.open(info_file, "w", **profile) as dst:
                    dst.write(cleaned.astype(data.dtype), 1)

                print(f"✅ Cleaned {suffix}: {info_file}")

✅ Cleaned length: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/channels_length.asc
✅ Cleaned depth: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/channels_depth.asc
✅ Cleaned distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/channels_distance.asc
✅ Cleaned width: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/channels_width.asc
✅ Cleaned length: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/5haStreams_length.asc
✅ Cleaned depth: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/5haStreams_depth.asc
✅ Cleaned distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/5haStreams_distance.asc
✅ Cleaned width: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/5haStreams_width.asc
✅ Cleaned length: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/channels_length.asc
✅ Cleaned depth: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/channels_depth.asc
✅ Cleaned distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/channels_distance.asc
✅ Cleaned width: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/channels_width.